In [ ]:
import os
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

In [ ]:
CLEAN_PATH = "../data/Chest Cancer Clean"

classes = [
    "Adenocarcinoma",
    "Large Cell Carcinoma",
    "Normal"
]

In [ ]:
image_paths = []
image_labels = []

for cls in classes:
    class_path = os.path.join(CLEAN_PATH, cls)

    for file in os.listdir(class_path):
        image_paths.append(os.path.join(class_path, file))
        image_labels.append(cls)

print("Total images:", len(image_paths))

In [ ]:
from sklearn.model_selection import train_test_split

image_paths = []
image_labels = []

for cls in classes:

    class_path = os.path.join(CLEAN_PATH, cls)

    for file in os.listdir(class_path):

        image_paths.append(
            os.path.join(class_path, file)
        )

        image_labels.append(cls)


X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths,
    image_labels,
    test_size=0.30,
    stratify=image_labels,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

In [ ]:
print("Training distribution:")
print(pd.Series(y_train).value_counts())

print("\nValidation distribution:")
print(pd.Series(y_val).value_counts())

print("\nTesting distribution:")
print(pd.Series(y_test).value_counts())

### Image Preprocessing

Medical images have different dimensions and color modes. Before training the CNN, all images are resized to a fixed resolution and converted to RGB.

The pixel values are normalized to the range [0, 1].

In [ ]:
IMG_SIZE = (224, 224)

def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image = image.resize(IMG_SIZE)

    image = np.array(image) / 255.0

    return image

In [ ]:
sample_image = preprocess_image(X_train[0])

print("Shape:", sample_image.shape)
print("Minimum:", sample_image.min())
print("Maximum:", sample_image.max())

### Data Augmentation

In [ ]:
import tensorflow as tf

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1)
])

In [ ]:
import tensorflow as tf
import sys

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)

In [ ]:
sample = tf.convert_to_tensor(
    preprocess_image(X_train[0]),
    dtype=tf.float32
)

augmented = data_augmentation(
    tf.expand_dims(sample, axis=0),
    training=True
)

print("Original shape:", sample.shape)
print("Augmented shape:", augmented.shape)

### A function to load images

In [ ]:
def load_image(image_path, label):
    image = preprocess_image(image_path)
    
    return image, label

In [ ]:
label_map = {
    "Adenocarcinoma": 0,
    "Large Cell Carcinoma": 1,
    "Normal": 2
}

y_train_encoded = [label_map[label] for label in y_train]
y_val_encoded = [label_map[label] for label in y_val]
y_test_encoded = [label_map[label] for label in y_test]

### Create TensorFlow datasets

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train_encoded)
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_val, y_val_encoded)
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (X_test, y_test_encoded)
)

In [ ]:
def load_image(image_path, label):
    image = tf.io.read_file(image_path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(image, IMG_SIZE)

    image = tf.cast(image, tf.float32)

    return image, label

In [ ]:
BATCH_SIZE = 16

train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train_encoded)
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_val, y_val_encoded)
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (X_test, y_test_encoded)
)

train_dataset = train_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_dataset = test_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train_encoded)
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_val, y_val_encoded)
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (X_test, y_test_encoded)
)

In [ ]:
train_dataset = train_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_dataset = test_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

In [ ]:
def augment_image(image, label):
    image = data_augmentation(
        tf.expand_dims(image, axis=0),
        training=True
    )

    image = tf.squeeze(image, axis=0)

    return image, label

In [ ]:
# Shuffle ONLY training data
train_dataset = train_dataset.shuffle(
    buffer_size=len(X_train),
    seed=42
)

# Augmentation ONLY for training
train_dataset = train_dataset.map(
    augment_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

In [ ]:
# Batch
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(
    tf.data.AUTOTUNE
)

val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(
    tf.data.AUTOTUNE
)

test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(
    tf.data.AUTOTUNE
)

In [ ]:
for images, labels in train_dataset.take(1):
    print("Images:", images.shape)
    print("Labels:", labels.shape)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_encoded),
    y=y_train_encoded
)

class_weights = dict(
    zip(
        np.unique(y_train_encoded),
        class_weights
    )
)

class_weights

# Give more importance to Adenocarcinoma
class_weights[0] = class_weights[0] * 1.5

print(class_weights)

In [ ]:
for class_name, class_id in label_map.items():
    print(class_name, ":", class_weights[class_id])

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

print("EfficientNetB0 imported successfully")

In [ ]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

### Build the EfficientNetB0 classifier

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(3, activation="softmax")
])

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

In [ ]:
EPOCHS = 20

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr]
)

In [ ]:
model.summary()

In [ ]:
import os

missing = [path for path in X_train + X_val + X_test if not os.path.exists(path)]

print("Missing images:", len(missing))

if missing:
    print(missing[:10])

In [ ]:
test_loss, test_accuracy = model.evaluate(test_dataset)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_true = []
y_pred = []

for images, labels in test_dataset:
    predictions = model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

print(classification_report(
    y_true,
    y_pred,
    target_names=[
        "Adenocarcinoma",
        "Large Cell Carcinoma",
        "Normal"
    ]
))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

In [ ]:
import os

os.makedirs("../models", exist_ok=True)

model.save("../models/chest_cancer_model.keras")

print("Chest model saved successfully.")

In [ ]:
model.save("../models/chest_cancer_model.keras")

In [ ]:
import tensorflow as tf
import numpy as np

IMG_SIZE = (224, 224)

class_names = [
    "Adenocarcinoma",
    "Large Cell Carcinoma",
    "Normal"
]

def predict_chest_image(image_path):

    image = tf.io.read_file(image_path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(image, IMG_SIZE)

    image = tf.cast(image, tf.float32)

    image = tf.expand_dims(image, axis=0)

    predictions = model.predict(image, verbose=0)[0]

    predicted_index = np.argmax(predictions)

    predicted_class = class_names[predicted_index]

    confidence = predictions[predicted_index]

    return predicted_class, confidence

In [ ]:
test_image = "../data/Chest Cancer Clean/Adenocarcinoma/000000 (6).png"

predicted_class, confidence = predict_chest_image(test_image)

print("Prediction:", predicted_class)
print("Confidence:", round(float(confidence) * 100, 2), "%")

In [ ]:
import os

test_folders = [
    "Adenocarcinoma",
    "Large Cell Carcinoma",
    "Normal"
]

for class_name in test_folders:

    folder_path = os.path.join(
        "../data/Chest Cancer Clean",
        class_name
    )

    test_file = os.listdir(folder_path)[0]
    test_image = os.path.join(folder_path, test_file)

    predicted_class, confidence = predict_chest_image(test_image)

    print("\nActual:", class_name)
    print("Image:", test_file)
    print("Prediction:", predicted_class)
    print("Confidence:", round(float(confidence) * 100, 2), "%")

In [ ]:
import os

base_path = "../data/Chest Cancer Clean"

for class_name in class_names:

    folder_path = os.path.join(base_path, class_name)

    files = os.listdir(folder_path)[:10]

    correct = 0

    print("\n==============================")
    print("Actual:", class_name)
    print("==============================")

    for file in files:

        image_path = os.path.join(folder_path, file)

        predicted_class, confidence = predict_chest_image(image_path)

        if predicted_class == class_name:
            correct += 1

        print(
            file,
            "→",
            predicted_class,
            round(float(confidence) * 100, 2), "%"
        )

    print(
        "Accuracy on these 10:",
        round(correct / len(files) * 100, 2), "%"
    )